# 02 · Limpieza y transformación

Aplicamos las decisiones del diario de hallazgos (notebook 01) a `bank-additional.csv` y `customer-details.xlsx`. Los datos limpios se guardan en `data/processed/`.

In [1]:
# Cargar los datos

import pandas as pd

ruta_bank = "../data/raw/bank-additional.csv"
ruta_customer = "../data/raw/customer-details.xlsx"

df_bank = pd.read_csv(ruta_bank)
hojas_customer = pd.read_excel(ruta_customer, sheet_name=None)

print(df_bank.shape)
for nombre, hoja in hojas_customer.items():
    print(nombre, hoja.shape)



(43000, 24)
2012 (20115, 7)
2013 (8965, 7)
2014 (14090, 7)


In [2]:
# Eliminar columnas sin valor para el análisis (ver diario de hallazgos)

cols_eliminar = ["Unnamed: 0", "latitude", "longitude"]

df_bank = df_bank.drop(columns=cols_eliminar)
print(df_bank.shape)

(43000, 21)


In [3]:
# Pasar todas las columnas de texto a minúsculas

cols_texto = ["job", "marital", "education", "contact", "poutcome", "y"]

for col in cols_texto:
    df_bank[col] = df_bank[col].str.lower()

In [4]:
# job: quitar el punto de "admin."

df_bank["job"] = df_bank["job"].str.replace(".", "", regex=False)

# education: cambiar el punto separador por guion bajo

df_bank["education"] = df_bank["education"].str.replace(".", "_", regex=False)

In [5]:
# Comprobar que los cambios se hicieron correctamente

for col in cols_texto:
    print(df_bank[col].value_counts(dropna=False))
    print("-" * 40)

job
admin            10873
blue-collar       9654
technician        7026
services          4162
management        3050
retired           1790
entrepreneur      1522
self-employed     1489
housemaid         1123
unemployed        1063
student            903
NaN                345
Name: count, dtype: int64
----------------------------------------
marital
married     25999
single      12105
divorced     4811
NaN            85
Name: count, dtype: int64
----------------------------------------
education
university_degree      12722
high_school             9925
basic_9y                6309
professional_course     5477
basic_4y                4356
basic_6y                2386
NaN                     1807
illiterate                18
Name: count, dtype: int64
----------------------------------------
contact
cellular     27396
telephone    15604
Name: count, dtype: int64
----------------------------------------
poutcome
nonexistent    37103
failure         4461
success         1436
Name: count,

In [6]:
# Columnas con coma decimal: cambiar "," por "." y pasar a float
cols_coma = ["cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

for col in cols_coma:
    df_bank[col] = df_bank[col].str.replace(",", ".", regex=False).astype(float)

print(df_bank[cols_coma].dtypes)



cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
dtype: object


In [7]:
# Comprobar que los valores son razonables
print(df_bank[cols_coma].describe().round(3))

       cons.price.idx  cons.conf.idx  euribor3m  nr.employed
count       42529.000      43000.000  33744.000    43000.000
mean           93.574        -40.509      3.617     5166.853
std             0.580          4.637      1.737       72.411
min            92.201        -50.800      0.634     4963.600
25%            93.075        -42.700      1.344     5099.100
50%            93.749        -41.800      4.857     5191.000
75%            93.994        -36.400      4.961     5228.100
max            94.767        -26.900      5.045     5228.100


In [8]:
# Separar la fecha por "-" y quedarnos con la parte del mes
meses_en_datos = df_bank["date"].str.split("-", expand=True)[1]
print(meses_en_datos.value_counts(dropna=False))    

1
noviembre     3603
octubre       3599
julio         3593
marzo         3589
abril         3582
febrero       3577
septiembre    3569
mayo          3547
junio         3538
agosto        3526
enero         3517
diciembre     3512
NaN            248
Name: count, dtype: int64


In [9]:
# Diccionario de traducción: mes en español a número de mes
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

fechas = df_bank["date"]
for mes, numero in meses.items():
    fechas = fechas.str.replace(mes, str(numero), regex=False)

print(fechas.head())

0      2-8-2019
1     14-9-2016
2     15-2-2019
3    29-11-2015
4     29-1-2017
Name: date, dtype: str


In [10]:
# Convertir a formato fecha
df_bank["date"] = pd.to_datetime(fechas, format="%d-%m-%Y")

print(df_bank["date"].dtype)
print(df_bank["date"].min(), "→", df_bank["date"].max())
print("Nulos:", df_bank["date"].isnull().sum())


datetime64[us]
2015-01-01 00:00:00 → 2019-12-31 00:00:00
Nulos: 248


In [11]:
# Nuevas columnas a partir de la fecha (las pide el enunciado)
df_bank["contact_month"] = df_bank["date"].dt.month.astype("Int64")
df_bank["contact_year"] = df_bank["date"].dt.year.astype("Int64")

print(df_bank[["date", "contact_month", "contact_year"]].head())
print(df_bank["contact_year"].value_counts(dropna=False).sort_index())

        date  contact_month  contact_year
0 2019-08-02              8          2019
1 2016-09-14              9          2016
2 2019-02-15              2          2019
3 2015-11-29             11          2015
4 2017-01-29              1          2017
contact_year
2015    8544
2016    8533
2017    8562
2018    8549
2019    8564
<NA>     248
Name: count, dtype: Int64


In [12]:
# Nulos en texto: rellenar con "unknown" (no inventamos categorías)
cols_categoricas = ["job", "marital", "education"]

for col in cols_categoricas:
    df_bank[col] = df_bank[col].fillna("unknown")

print(df_bank[cols_categoricas].isnull().sum())


job          0
marital      0
education    0
dtype: int64


In [13]:
# Ver cómo se reparten las binarias antes de decidir
cols_binarias = ["default", "housing", "loan"]

for col in cols_binarias:
    print(df_bank[col].value_counts(dropna=False))
    print("-" * 40)

default
0.0    34016
NaN     8981
1.0        3
Name: count, dtype: int64
----------------------------------------
housing
1.0    22498
0.0    19476
NaN     1026
Name: count, dtype: int64
----------------------------------------
loan
0.0    35442
1.0     6532
NaN     1026
Name: count, dtype: int64
----------------------------------------


In [14]:
# Comprobar si los nulos de housing y loan coinciden
ambos_nulos = df_bank["housing"].isnull() & df_bank["loan"].isnull()
print("Nulos en ambas columnas:", ambos_nulos.sum())

# Binarias: no se rellenan (sería inventar), solo se pasan a entero con nulos
for col in cols_binarias:
    df_bank[col] = df_bank[col].astype("Int64")

print(df_bank[cols_binarias].dtypes)


Nulos en ambas columnas: 1026
default    Int64
housing    Int64
loan       Int64
dtype: object


In [15]:
# Mirar la edad antes de decidir
print(df_bank["age"].describe().round(1))
print("Media: ", round(df_bank["age"].mean(), 1))
print("Mediana: ", df_bank["age"].median())




count    37880.0
mean        40.0
std         10.4
min         17.0
25%         32.0
50%         38.0
75%         47.0
max         98.0
Name: age, dtype: float64
Media:  40.0
Mediana:  38.0


In [16]:
# age: rellenar con la mediana (no le afectan los valores extremos)
mediana_edad = df_bank["age"].median()

df_bank["age"] = df_bank["age"].fillna(mediana_edad).astype(int)

print(f"Mediana usada: {mediana_edad}")
print("Nulos en age:", df_bank["age"].isnull().sum())
print(df_bank["age"].dtype)

Mediana usada: 38.0
Nulos en age: 0
int64


In [17]:
# Indicadores económicos sin nulos: definen el "periodo económico"
cols_periodo = ["emp.var.rate", "cons.conf.idx", "nr.employed"]

# ¿Cuántos valores distintos de cons.price.idx hay en cada periodo?
ipc_por_periodo = df_bank.groupby(cols_periodo)["cons.price.idx"].nunique()
print(f"Máximo de valores de IPC distintos en un periodo: {ipc_por_periodo.max()}")

# ¿Y de euribor3m?
euribor_por_periodo = df_bank.groupby(cols_periodo)["euribor3m"].agg(["nunique", "min", "max"])
print(euribor_por_periodo.head(8))


Máximo de valores de IPC distintos en un periodo: 1
                                        nunique    min    max
emp.var.rate cons.conf.idx nr.employed                       
-3.4         -30.1         5017.5             7  0.714  0.722
             -29.8         5017.5            21  0.739  0.819
             -26.9         5017.5            17  0.720  0.754
-3.0         -33.0         5023.5            13  0.700  0.721
-2.9         -40.8         5076.2            13  1.099  1.286
             -33.6         5076.2            16  0.908  1.085
             -31.4         5076.2            17  0.821  0.884
-1.8         -50.0         5099.1            21  1.510  1.811


In [18]:
# cons.price.idx: es único por periodo → se recupera el valor exacto
ipc_del_periodo = df_bank.groupby(cols_periodo)["cons.price.idx"].transform("first")
df_bank["cons.price.idx"] = df_bank["cons.price.idx"].fillna(ipc_del_periodo)

# euribor3m: varía muy poco dentro del periodo → mediana de su periodo
euribor_del_periodo = df_bank.groupby(cols_periodo)["euribor3m"].transform("median")
df_bank["euribor3m"] = df_bank["euribor3m"].fillna(euribor_del_periodo)

print(df_bank[["cons.price.idx", "euribor3m"]].isnull().sum())
print(df_bank["euribor3m"].describe().round(3))



cons.price.idx    0
euribor3m         0
dtype: int64
count    43000.000
mean         3.616
std          1.737
min          0.634
25%          1.344
50%          4.857
75%          4.961
max          5.045
Name: euribor3m, dtype: float64


In [19]:
# y: "yes" → 1, "no" → 0 (así su media es la tasa de éxito)
df_bank["y"] = df_bank["y"].map({"yes": 1, "no": 0})

print(df_bank["y"].value_counts())
print(f"Tasa de éxito: {round(df_bank['y'].mean() * 100, 2)} %")




y
0    38156
1     4844
Name: count, dtype: int64
Tasa de éxito: 11.27 %


In [20]:
# Comprobación final: tipos y nulos
df_bank.info()


<class 'pandas.DataFrame'>
RangeIndex: 43000 entries, 0 to 42999
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   age             43000 non-null  int64         
 1   job             43000 non-null  str           
 2   marital         43000 non-null  str           
 3   education       43000 non-null  str           
 4   default         34019 non-null  Int64         
 5   housing         41974 non-null  Int64         
 6   loan            41974 non-null  Int64         
 7   contact         43000 non-null  str           
 8   duration        43000 non-null  int64         
 9   campaign        43000 non-null  int64         
 10  pdays           43000 non-null  int64         
 11  previous        43000 non-null  int64         
 12  poutcome        43000 non-null  str           
 13  emp.var.rate    43000 non-null  float64       
 14  cons.price.idx  43000 non-null  float64       
 15  cons.conf.idx

In [21]:
# Nulos que quedan
nulos_finales = df_bank.isnull().sum()
print(nulos_finales[nulos_finales > 0])

default          8981
housing          1026
loan             1026
date              248
contact_month     248
contact_year      248
dtype: int64


## Registro de limpieza

| Columna | Qué se hizo | Por qué | Resultado |
|---|---|---|---|
| Unnamed: 0, latitude, longitude | Eliminadas | Índice sin valor; coordenadas de EE. UU., no de Portugal | 24 → 21 columnas |
| job, marital, education, contact, poutcome, y | Pasadas a minúsculas | Criterio único en todo el texto | Mismos recuentos |
| job, education | Puntos eliminados o cambiados por "_" | Legibilidad (admin., university.degree) | Mismos recuentos |
| cons.price.idx, cons.conf.idx, euribor3m, nr.employed | "," → "." y a float | Decimal con coma leído como texto | 4 columnas numéricas |
| date | Meses traducidos con diccionario y convertida a fecha | Mes escrito en español | datetime, 2015–2019, 248 nulos |
| contact_month, contact_year | Creadas desde date (Int64) | Las pide el enunciado | ~8.500 llamadas por año |
| job, marital, education | Nulos → "unknown" | No inventar categorías | 0 nulos |
| housing, loan | Nulos sin rellenar; a Int64 | Los 1.026 nulos coinciden en las mismas filas; rellenar sería inventar | 0/1 con <NA> |
| default | Nulos sin rellenar; a Int64 | Solo 3 impagos (0,007 %): sin valor para el análisis | 0/1 con <NA> |
| age | Nulos → mediana (38) y a int | La media (40) está inflada por clientes mayores (máx. 98); la mediana no | 0 nulos |
| cons.price.idx | Nulos → valor de su periodo económico (groupby + transform "first") | El IPC es único en cada combinación de emp.var.rate, cons.conf.idx y nr.employed: se recupera el dato real | 471 nulos recuperados |
| euribor3m | Nulos → mediana de su periodo económico (groupby + transform "median") | El euríbor apenas varía dentro de un periodo; la mediana global (4,857) sería muy imprecisa | 9.256 nulos rellenados; media 3,617 → 3,616 |
| date, contact_month, contact_year | 248 nulos sin rellenar | El día exacto no se puede deducir; las filas tienen el resto de datos útiles | Se excluyen solo en el análisis temporal |
| y | "yes"/"no" → 1/0 con .map() | Su media es directamente la tasa de éxito | Tasa de éxito: 11,27 % |